# 1b. The author's words: more real handwriting from the writing pages

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chingkheinganba231005/Meitei-Mayek-Diffusion-model/blob/main/notebooks/1b_author_words.ipynb)

**Runtime: CPU.** A few minutes. Run any time before notebook 4.

The word-recognition project's writing pages (`real/real_words_pages.pdf`, 29 pages, 500 items:
words of its test lexicon, so never training words) were written up to item 100. Write more of
them in the same way: on the iPad, in Markup, with the same pen, one word in each box; page 6
still has items 101-106 empty, and pages 7-29 hold items 107-500. Export the written pages as
PDF into `WORK/real/written/` (any file names; a page given twice counts once, the later one).
As many as time allows: every word helps, up to 500 in all.

1. **Cut** the new pages into words (`scripts/cut_writing_pages.py` of the word repository) and
   draw a contact sheet per page: each printed word beside the handwriting cut out for it.
   **Look at every sheet**: a word written differently from its print (a missing full stop, a
   different letter) goes into `WORK/real/corrections.json` as
   `{"000123.png": {"text": "...", "reason": "..."}}`, and this notebook is run again.
2. **Merge** them with the first 100 words (the word repository's `real_words/`, with its
   corrections) into `WORK/real_words/`, which notebooks 4 and 6 use.

The words are the author's own handwriting, released like the first 100 under CC BY 4.0.

In [ ]:
import os

# Where things are. Change these to match your Drive.
WORK = "/content/drive/MyDrive/meitei-word-diffusion"         # this project's working folder
WORK_OLD = "/content/drive/MyDrive/meitei-word-recognition"   # the word-recognition project's folder
REPO = "chingkheinganba231005/Meitei-Mayek-Diffusion-model"
BRANCH = "main"
WORD_REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
WORD_COMMIT = "bdee9e7"                                       # the synthesiser this project builds on
CONTENT = "/content"                                          # the runtime's own disk

TEMPLATE = f"{WORK_OLD}/real/real_words_pages.pdf"     # the blank writing pages
WRITTEN = f"{WORK}/real/written"                         # the newly written pages (PDF)

DISCONNECT_AT_END = True    # the last cell disconnects the runtime (Colab bills a connected runtime)

In [ ]:
import json, re, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
os.chdir(CONTENT)
CODE, WORD = f"{CONTENT}/code", f"{CONTENT}/word_repo"


def run(*args):
    """Runs a command, shows its output, and stops the notebook if it fails."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    if (r.stdout + r.stderr).strip():
        print(r.stdout + r.stderr)
    if r.returncode:
        raise SystemExit(f"exit code {r.returncode}: {' '.join(str(a) for a in args)[:200]}")


def stream(*args):
    """Runs a long command, showing its output as it comes; stops the notebook if it fails.
    Returns the output."""
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    if p.wait():
        raise SystemExit(f"exit code {p.returncode}: {' '.join(str(a) for a in args)[:200]}")
    return "".join(lines)


def clone(url, path, ref):
    """A fresh copy of a repository at a branch or a commit."""
    shutil.rmtree(path, ignore_errors=True)
    run("git", "clone", "-q", url, path)
    run("git", "-C", path, "checkout", "-q", ref)


clone(f"https://github.com/{REPO}.git", CODE, BRANCH)
clone(f"https://github.com/{WORD_REPO}.git", WORD, WORD_COMMIT)
sys.path[:0] = [CODE, WORD]
os.chdir(CODE)

for d in ("data", "runs", "results", "eval"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
run("git", "-C", CODE, "log", "-1", "--format=code: %h %s")

from glob import glob
from IPython.display import Image as Show, display
from PIL import features
run(sys.executable, "-m", "pip", "install", "-q", "pymupdf")
if not features.check("raqm"):     # Pillow shapes Meitei Mayek with libraqm, which needs FriBiDi
    run("apt-get", "-qq", "install", "-y", "libfribidi0")
    raise SystemExit("installed FriBiDi for Pillow: Runtime > Restart session, then run the cells again")
pdfs = sorted(glob(f"{WRITTEN}/*.pdf"))
if not os.path.exists(TEMPLATE):
    raise SystemExit(f"{TEMPLATE} is missing: the blank pages of the word-recognition project")
if not pdfs:
    raise SystemExit(f"no written pages in {WRITTEN}: export them there from the iPad first")
print(f"{len(pdfs)} file(s) of written pages")

## 1. Cut the new pages

In [ ]:
CUT, NEW = f"{CONTENT}/real/cut", f"{CONTENT}/real/new"
corr_new = f"{WORK}/real/corrections.json"
run(sys.executable, f"{WORD}/scripts/cut_writing_pages.py", "--template", TEMPLATE, "--written", *pdfs,
    "--out-dir", CUT, "--sheets", f"{WORK}/real/sheets", "--summary", f"{WORK}/results/author_pages_cut.json")
args = ["--corrections", corr_new] if os.path.exists(corr_new) else []
shutil.rmtree(NEW, ignore_errors=True)
run(sys.executable, f"{WORD}/scripts/release_real_set.py", "--set", CUT, "--out", NEW, *args)
sheets = sorted(Path(f"{WORK}/real/sheets").glob("*.png"))
print(f"{len(sheets)} contact sheets in {WORK}/real/sheets; the first:")
if sheets:
    display(Show(str(sheets[0])))

## 2. Merge with the first 100 words

In [ ]:
from collections import Counter
from mayek_diffusion.owner import merge_words, read_words

ALL = f"{WORK}/real_words"
shutil.rmtree(ALL, ignore_errors=True)
n = merge_words(f"{WORD}/real_words", NEW, ALL)
shutil.copyfile(f"{WORD}/real_words/LICENSE", f"{ALL}/LICENSE")
names, texts, _ = read_words(ALL)
counts = Counter(c for t in texts for c in t)
summary = {"words": n, "characters": sum(counts.values()), "distinct_characters": len(counts),
           "rarest": sorted(counts.items(), key=lambda kv: kv[1])[:8]}
Path(f"{WORK}/results/author_words.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False))
print(json.dumps(summary, ensure_ascii=False))

## Disconnect

Colab bills a connected runtime whether or not it is working. The cell below disconnects it
(`DISCONNECT_AT_END = True`); otherwise use *Runtime > Disconnect and delete runtime*. Every
result is already on Drive.

In [ ]:
# Colab bills a connected runtime whether or not it is working: disconnect it now.
if DISCONNECT_AT_END:
    from google.colab import runtime
    runtime.unassign()
else:
    print("Finished. Disconnect the runtime: Runtime > Disconnect and delete runtime.")